<a href="https://colab.research.google.com/github/nangmrr752/virgo-models/blob/claude%2Foptimistic-noether-sh22bx/chat/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Train Virgo 1.0 (one click)

**Before you start (once):**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. On Hugging Face, open [google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it) and click **Agree and access**.
3. Get a token: [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) → **Create new token** → type **Write** → copy it.

**Then:** menu **Runtime → Run all**. Paste your token when asked. That's it. Keep this tab open (~30–60 min).

Tip: set `QUICK_TEST = True` below for a fast first try (~10–15 min).

In [ ]:
#@title ⚙️ Settings
QUICK_TEST = True  #@param {type:"boolean"}
BASE_MODEL = "google/gemma-3-4b-it"  #@param ["google/gemma-3-4b-it", "google/gemma-3-1b-it"]
SAVE_TO_HUGGING_FACE = True  #@param {type:"boolean"}

In [ ]:
#@title 1️⃣ Set up (install + sign in)
import os, subprocess, sys, torch
if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU. Menu Runtime → Change runtime type → T4 GPU, then Run all again.")
print("✅ GPU:", torch.cuda.get_device_name(0))

# Download Virgo's code (a public repo needs no token; a private one asks for a GitHub token).
os.chdir("/content")
subprocess.run(["rm", "-rf", "virgo-models"])
REPO = "github.com/nangmrr752/virgo-models"
if subprocess.run(["git", "clone", "-q", f"https://{REPO}"]).returncode != 0:
    gh = None
    try:
        from google.colab import userdata
        gh = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass
    if not gh:
        from getpass import getpass
        gh = getpass("The repo is private. Paste a GitHub token (github_pat_...): ").strip()
    r = subprocess.run(["git", "clone", "-q", f"https://x-access-token:{gh}@{REPO}"], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit("❌ Couldn't download virgo-models: " + r.stderr.replace(gh, "***"))
os.chdir("/content/virgo-models")
print("✅ Downloaded Virgo's code")

# Only what chat training needs (works on Colab's Python 3.13).
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("❌ Install failed: send the lines above to Claude.")
print("✅ Installed")

from huggingface_hub import login
token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass
if not token:
    from getpass import getpass
    token = getpass("Paste your Hugging Face token (hf_...) and press Enter: ").strip()
login(token=token)
print("✅ Signed in to Hugging Face")

In [ ]:
#@title 2️⃣ Train Virgo
epochs = "1" if QUICK_TEST else "3"
!python scripts/check_data.py
!python chat/train.py --base {BASE_MODEL} --epochs {epochs}

In [ ]:
#@title 3️⃣ Score Virgo
!python chat/evaluate.py --base {BASE_MODEL}

In [ ]:
#@title 4️⃣ Try Virgo-1.0-Angkor
import sys; sys.path.append("chat")
from virgo_chat import VirgoChat
bot = VirgoChat(BASE_MODEL, adapter="chat/out/virgo-1.0-chat-lora")
for q in ["Who are you?", "អ្នកជានរណា?", "What is 15% of 200?", "ពន្យល់ AI ឱ្យងាយយល់"]:
    print("🧑", q)
    print("🤖", bot.reply([{"role": "user", "content": q}], temperature=0), "\n")

In [ ]:
#@title 5️⃣ Save Virgo-1.0-Angkor to your Hugging Face account (private)
MODEL_NAME = "Virgo-1.0-Angkor"  #@param {type:"string"}
if SAVE_TO_HUGGING_FACE:
    from huggingface_hub import HfApi
    api = HfApi()
    repo = f"{api.whoami()['name']}/{MODEL_NAME}"
    card = f"""---
base_model: {BASE_MODEL}
library_name: peft
license: gemma
language: [en, km]
tags: [virgo, ksn, khmer, lora]
---
# {MODEL_NAME}

**{MODEL_NAME}** is KSN's own assistant model for [Virgo AI](https://virgoai.camksn.com): friendly, clear and honest, in English and Khmer.

It is a LoRA fine-tune of Google's [{BASE_MODEL}](https://huggingface.co/{BASE_MODEL}), trained on Virgo's own conversations (identity, everyday help, Khmer, customer support, answer style).
Use is subject to the [Gemma Terms of Use](https://ai.google.dev/gemma/terms).

Made by KSN · https://soknang.camksn.com/
"""
    open("chat/out/virgo-1.0-chat-lora/README.md", "w", encoding="utf-8").write(card)
    api.create_repo(repo, private=True, exist_ok=True)
    api.upload_folder(folder_path="chat/out/virgo-1.0-chat-lora", repo_id=repo, commit_message=f"{MODEL_NAME}")
    print(f"✅ Saved {MODEL_NAME}: https://huggingface.co/{repo}")
print("🎉 Done! Send the score table and Virgo's answers above to Claude.")